# Composite urban index: advisor's method (A) vs corrected method (B)

This notebook builds both indices for the **96 São Paulo districts and 77 Chicago Community Areas (173 units)** and runs the pre-registered tests that compare them. It calls the same functions as `analysis/scripts/build_simple_index_ab.py`, which produced the [methodology report](results/SP_CHI/simple_index_ab_2026_10_01/README.md); the last section checks that this notebook reproduces the published tables exactly.

**Rules of evidence used throughout**

- Every number shown is computed here from repository data. Nothing is typed in by hand.
- The definitions, tests and pass/fail thresholds were fixed in protocol **P-AB-1** ([docs/SIMPLE_INDEX.md](../docs/SIMPLE_INDEX.md)) *before* any A or B value existed, so the thresholds could not be tuned to the result.
- Construction checks stop the run if any input is lost, double-counted or misassigned.

**Run time:** about 2 minutes; peak memory about 1.7 GB. Run the cells in order from the repository's `.venv` kernel.

## 0. Setup

The functions live in `analysis/scripts/build_simple_index_ab.py`. Importing them (instead of copying code) guarantees that the notebook and the published report use one definition of each factor and test.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis/scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis/scripts'))
import build_simple_index_ab as ab

pd.set_option('display.width', 200, 'display.max_columns', 30)
sp = gpd.read_parquet(ROOT / 'analysis/work/prepared/SP/sp_prep_2026_09_10_v3/N02/districts.parquet')
chi = gpd.read_parquet(ROOT / 'analysis/work/prepared/Chicago/chi_local_2026_09_16_v1/districts.parquet')
NAMES = {**{'SP:' + i: n for i, n in zip(sp.district_id, sp.nm_distrito_municipal)}, **dict(zip(chi.unit_id, chi.district_name))}
named = lambda df: df.assign(name=df.index.map(NAMES))

## 1. Units and land area

**Units.** São Paulo municipal districts (`SP:01`–`SP:96`, Brás = `SP:10`) and Chicago Community Areas (`CHI:01`–`CHI:77`).

**Land area** = unit area minus municipal hydrography (rivers, reservoirs, Lake Michigan). It is used only by method B, as the denominator of the count factors.

Why land area matters: the units differ in size by two orders of magnitude, so a raw count partly measures *how big the unit is*.

In [2]:
U = ab.units()
pd.DataFrame({c: u.land_km2.describe()[['count', 'min', '50%', 'max']] for c, u in U.items()}).round(3)

,SP,Chicago
count,96.000,77.000
min,2.148,1.570
50%,9.632,7.170
max,206.876,33.958


## 2. The six factors

Each factor is built once per unit as **additive components** (counts, sums and land), so that any group of units can later be recombined exactly (needed for test T1).

| Factor | What it represents | Method A (advisor, as specified) | Method B (corrected) | Source |
|---|---|---|---|---|
| **C** commercial establishments | business activity | number of places | places ÷ land km² | Overture Places 2026-08-19.0, taxonomy top level shopping, food_and_drink, services_and_business, lifestyle_services, lodging; no filter (same rule in both cities) |
| **R** residential establishments | housing | number of dwellings | dwellings ÷ land km² | SP: IBGE CNEFE 2022 private dwellings. Chicago: Census 2020 housing units, blocks cut by a boundary split by area share |
| **H** transportation hubs | public transport | stations + bus stops | (stations + bus stops) ÷ land km² | CTA and SPTrans GTFS. Stations: CTA `L` parent stations, SP metro stop records grouped by name. CPTM excluded (no Metra in the data) |
| **V** building height | vertical form | mean Overture `height` of buildings with a height | GHSL built volume ÷ built surface | Overture: LiDAR-based in Chicago, OSM tags in SP. GHSL: one satellite method in both cities |
| **N** number of streets | network extent | number of road segments | segments ÷ land km² | Overture road segments, ten street classes, each assigned whole by its midpoint |
| **L** average street length | network grain | mean segment length | same as A | as N |

Why these sources: the [source audit](results/SP_CHI/simple_index_source_audit_2026_10_01/README.md) and the [places coverage test](results/SP_CHI/places_cnefe_coverage_2026_10_01/README.md), where Overture matched 20.10% of São Paulo's census-recorded establishment addresses by name and OpenStreetMap 2.35%.

The next cell builds all components and runs the construction checks; a failed check raises an error and stops the notebook.

In [3]:
comp = ab.build_components()
named(comp).head()

check ok commercial_independent_route_SP


check ok commercial_matches_source_audit_SP


check ok dwellings_SP_conserved


check ok heights_A_SP_reconcile_audit


built SP


check ok commercial_independent_route_Chicago


... (6 outputs left out)

built Chicago


check ok heights_B_recombination_exact


check ok units_complete


,city,land_km2,C_count,R_count,H_stations,H_bus,H_bus_merged,V_A_sum,V_A_n,N_count,L_sum,V_B_num,V_B_den,name
unit_id,,,,,,,,,,,,,,
SP:01,SP,7.168585,2967,41132.0,0.0,165,164,144368.3620,26627,1835,152513.687755,4.356764e+07,3.391655e+06,AGUA RASA
SP:02,SP,7.306681,1497,17911.0,0.0,114,112,95172.8749,14071,1659,126520.851845,3.806728e+07,2.575029e+06,ALTO DE PINHEIROS
SP:03,SP,33.064820,693,30332.0,0.0,196,174,71284.3740,14684,1441,200108.190453,2.772119e+07,2.579387e+06,ANHANGUERA
SP:04,SP,6.897900,1997,37218.0,0.0,163,162,186587.7460,36464,1825,155590.024305,3.371950e+07,3.163850e+06,ARICANDUVA
SP:05,SP,6.492532,1899,40974.0,1.0,245,227,129438.3460,27234,1384,135565.531363,3.238514e+07,2.767419e+06,ARTUR ALVIM


### Construction checks

Each check compares a quantity built here with an independent count of the same thing:

- commercial counts by two different point-in-polygon routes, and against the earlier source audit;
- São Paulo dwellings against the CNEFE total; Chicago housing units allocated plus the remainder outside the city against the block total, and the same block weights reproducing the published population allocation;
- buildings with a height assigned to units against the audit totals;
- GHSL height recombined from volume and surface against the stored values;
- 173 units, no missing value.

In [4]:
pd.DataFrame(ab.checks).T

,passed,detail
commercial_independent_route_SP,True,sjoin vs contains_xy counts per unit
commercial_matches_source_audit_SP,True,per-unit totals vs audit table
dwellings_SP_conserved,True,sum 4992162 vs CNEFE private dwellings 4992162
hubs_SP_inventory,True,"{'stations_in_feed': 100, 'stations_in_units':..."
heights_A_SP_reconcile_audit,True,assigned 2080386 vs audit 2080386 (diff 0; poi...
commercial_independent_route_Chicago,True,sjoin vs contains_xy counts per unit
commercial_matches_source_audit_Chicago,True,per-unit totals vs audit table
dwellings_CHI_blocks_joined,True,0 pieces without HOUSING20
dwellings_CHI_weights_reproduce_U3,True,area weights reproduce published U3 population
dwellings_CHI_no_overallocation,True,max weight sum per block 1.0000000000163682


In [5]:
comp.groupby('city')[['land_km2', 'C_count', 'R_count', 'H_stations', 'H_bus', 'V_A_n', 'N_count']].sum().round(1)

,land_km2,C_count,R_count,H_stations,H_bus,V_A_n,N_count
city,,,,,,,
Chicago,583.8,73672,1262056.0,123.0,9867,749136,53329
SP,1510.2,301655,4992162.0,100.0,21978,2080386,190348


## 3. Scaling and the index

Both methods use the advisor's scaling: each factor is **min–max scaled over the 173 units pooled**,

$$x' = \frac{x - \min x}{\max x - \min x} \in [0, 1],$$

and the **index is the sum of the six scaled factors** (equal weights, range 0–6). Built environment (C, R, H, V) is therefore 4/6 of the index and network (N, L) 2/6.

The only differences between A and B are the ones in the factor table: counts per unit (A) versus counts per km² of land (B), and the height source.

In [6]:
out, res, S1 = ab.build_indices(comp)
res = named(res)
cols = ['name', 'city', 'land_km2', 'A_index', 'A_rank', 'B_index', 'B_rank']
print('Top 10, method A'); display(res.sort_values('A_rank')[cols].head(10).round(3))
print('Top 10, method B'); display(res.sort_values('B_rank')[cols].head(10).round(3))

check ok scaled_range_A


check ok index_is_sum_A


check ok scaled_range_B


check ok index_is_sum_B


Top 10, method A


,name,city,land_km2,A_index,A_rank,B_index,B_rank
unit_id,,,,,,,
SP:30,GRAJAU,SP,92.417,3.380,1,0.731,168
SP:35,ITAIM BIBI,SP,9.866,2.907,2,2.551,7
SP:68,SACOMA,SP,14.489,2.650,3,1.901,34
SP:38,JABAQUARA,SP,13.989,2.642,4,1.956,26
SP:43,JARDIM ANGELA,SP,36.277,2.427,5,1.130,127
SP:71,SANTO AMARO,SP,15.759,2.390,6,1.627,62
CHI:32,LOOP,Chicago,3.969,2.256,7,3.432,3
SP:46,JARDIM SAO LUIS,SP,25.868,2.255,8,1.306,102
SP:37,ITAQUERA,SP,14.644,2.197,9,1.950,28


Top 10, method B


,name,city,land_km2,A_index,A_rank,B_index,B_rank
unit_id,,,,,,,
SP:66,REPUBLICA,SP,2.395,1.522,37,4.686,1
SP:78,SE,SP,2.148,1.108,76,3.713,2
CHI:32,LOOP,Chicago,3.969,2.256,7,3.432,3
SP:07,BELA VISTA,SP,2.718,1.152,69,3.272,4
CHI:08,NEAR NORTH SIDE,Chicago,6.837,2.074,16,2.835,5
SP:69,SANTA CECILIA,SP,3.759,1.109,74,2.746,6
SP:35,ITAIM BIBI,SP,9.866,2.907,2,2.551,7
SP:90,VILA MARIANA,SP,8.576,1.850,22,2.494,8
SP:26,CONSOLACAO,SP,3.815,1.092,79,2.476,9


### Worked example: Brás and Grajaú

Grajaú is a large peripheral district; Brás is a small dense central one. Compare their raw factors under both methods.

In [7]:
raw_cols = [c for c in res.columns if c.startswith(('A_raw_', 'B_raw_'))]
ex = res.loc[['SP:10', 'SP:30'], ['name', 'land_km2'] + raw_cols + ['A_rank', 'B_rank']].T
ex.loc['land ratio Grajaú/Brás'] = [None, res.land_km2['SP:30'] / res.land_km2['SP:10']]
ex

unit_id,SP:10,SP:30
name,BRAS,GRAJAU
land_km2,3.629273,92.416556
A_raw_C,4649,3437
A_raw_R,19723.0,154175.0
A_raw_H,72.0,644.0
A_raw_V,8.235518,4.179012
A_raw_N,658,6089
A_raw_L,98.348494,104.549058
B_raw_C,1280.972828,37.190306
B_raw_R,5434.421832,1668.261696


## 4. Tests of method bias (protocol P-AB-1)

"Bias" is defined operationally: **a measure of urban intensity should not change merely because unit boundaries are drawn differently, and should not rise mechanically with unit size.** The tests and their criteria were fixed before any result:

| Test | Question | Criterion fixed in advance |
|---|---|---|
| **T1** boundary invariance (decisive) | When neighbouring districts are reported as one unit, is the merged value within the range of its parts? | boundary-invariant if true for ≥ 95% of merged units |
| **T2** size dependence | Does the factor or index correlate with land area? | size-dependent if the 95% bootstrap interval of Spearman's ρ excludes 0 and \|ρ\| ≥ 0.3 |
| **T3** agreement | Do A and B rank the units the same way? | substantively disagree if Spearman < 0.8 or top-10 overlap < 7 |
| **T4** city composition | How many Chicago units are in the top and bottom 20? | descriptive, against Chicago's share of units (77/173) |
| **S1** sensitivity | Does merging paired bus stops (same name within 50 m) change the result? | descriptive |

### T1 — boundary invariance

**Intuition.** Reporting two neighbouring districts as one area changes nothing on the ground. An intensity measure of the merged area should therefore sit between the values of its parts: it cannot be more intense than its most intense part.

**Procedure.** The 96 SP districts are merged into their subprefeituras (`analysis/config/sp_district_subprefeitura.csv`); 29 of the 32 contain two or more districts. Every factor is recomputed for the merged units by the same definition (sum counts and land, recombine means from their sums). The index of a merged unit uses the 173-unit min–max unchanged.

**Why the outcome is a mathematical certainty, and the data measures its size.**

- *Count* (method A): $x_{\text{merged}} = \sum_i x_i$, which exceeds $\max_i x_i$ whenever two or more parts are positive.
- *Ratio of sums* (method B, and the means V and L): $\frac{\sum_i a_i}{\sum_i b_i}$ is a weighted average of the parts' ratios $\frac{a_i}{b_i}$ with weights $\frac{b_i}{\sum b}$, so it always lies between the smallest and largest part.

Column `merged_over_max_part_median` shows how many times the merged value exceeds the largest part.

In [8]:
t1, t1_detail = ab.test_t1(comp, out)
t1

,method,factor,subprefeituras_tested,within_parts,share_within,verdict,merged_over_max_part_median,merged_over_max_part_max
0,A,C,29,0,0.0000,depends on boundaries,2.3183,4.5485
1,A,R,29,0,0.0000,depends on boundaries,2.4890,5.1785
2,A,H,29,0,0.0000,depends on boundaries,2.2936,5.6786
3,A,V,29,29,1.0000,boundary-invariant,0.9521,1.0000
4,A,N,29,0,0.0000,depends on boundaries,2.4111,6.6125
5,A,L,29,29,1.0000,boundary-invariant,0.9311,0.9924
6,A,index,29,0,0.0000,depends on boundaries,2.1408,4.2938
7,B,C,29,29,1.0000,boundary-invariant,0.6902,0.9932
8,B,R,29,29,1.0000,boundary-invariant,0.8234,0.9679
9,B,H,29,29,1.0000,boundary-invariant,0.8129,0.9939


Merged units whose value fell outside the range of their parts, method B. A sum of separately scaled factors can fall just outside the range of the parts' sums even when every factor is within range:

In [9]:
t1_detail[(t1_detail.method == 'B') & ~t1_detail.within]

,method,factor,subprefeitura_id,merged,parts_min,parts_max,within
26,B,index,27.0,0.52658,0.550698,1.0,False


### T2 — size dependence

**Intuition.** If a factor tracks unit size, the index partly ranks units by area. T2 measures the Spearman rank correlation of each factor and of the index with land area, pooled and within each city, with a 95% bootstrap interval (2,000 resamples, seed 20261001).

**How to read it together with T1.** Real urban structure can also correlate with size (large districts are often peripheral and less dense), so T2 alone cannot prove a method biased. A positive correlation in A combined with the T1 failure shows a mechanical size effect; a negative correlation in B is consistent with less dense peripheries, but T2 alone cannot prove it reflects urban form.

In [10]:
t2 = ab.test_t2(comp, out)
t2.pivot_table(index=['factor'], columns=['method', 'scope'], values='spearman').round(3)

method       A                     B              
scope  Chicago     SP pooled Chicago     SP pooled
factor                                            
C        0.378 -0.167  0.282  -0.170 -0.722 -0.163
H        0.595  0.731  0.728  -0.232 -0.378 -0.205
L       -0.043  0.147 -0.248  -0.043  0.147 -0.248
N        0.799  0.695  0.754  -0.130 -0.450  0.009
R        0.422  0.600  0.582  -0.225 -0.421 -0.044
V       -0.181 -0.322 -0.272  -0.333 -0.336 -0.033
index    0.610  0.635  0.650  -0.300 -0.583 -0.156

In [11]:
t2[t2.factor == 'index']

,method,factor,scope,n,spearman,ci_low,ci_high,verdict
18,A,index,pooled,173,0.6496,0.5392,0.7401,size-dependent
19,A,index,SP,96,0.6350,0.4656,0.7687,size-dependent
20,A,index,Chicago,77,0.6100,0.4186,0.7482,size-dependent
39,B,index,pooled,173,-0.1563,-0.3023,0.0046,not size-dependent
40,B,index,SP,96,-0.5831,-0.7341,-0.3892,size-dependent
41,B,index,Chicago,77,-0.3002,-0.5011,-0.0727,size-dependent


### T3 — agreement between A and B

If both methods measured the same thing, they would rank the units similarly. Brás is the project's reference district.

In [12]:
t3 = ab.test_t3(res, comp)
t3['unit_with_max_shift_name'] = NAMES[t3['unit_with_max_shift']]
pd.Series(t3)

spearman_pooled                             0.5521
spearman_SP                                 0.0892
spearman_Chicago                            0.4897
top10_overlap                                    2
bras_rank_A                                    106
bras_rank_B                                     23
max_rank_shift                                 167
unit_with_max_shift                          SP:30
verdict                     substantively disagree
unit_with_max_shift_name                    GRAJAU
dtype: object

### T4 — city composition

Chicago has 77 of the 173 units, so about 8.9 Chicago units would appear in a top or bottom 20 if city did not matter. Neither method can tell from available data whether a city imbalance reflects real differences or unequal sources (Overture Places provenance differs by city; dwellings come from two different censuses).

In [13]:
ab.test_t4(res, comp)

,method,group,chicago,sp,expected_chicago_at_unit_share
0,A,top20,2,18,8.9
1,A,bottom20,19,1,8.9
2,B,top20,2,18,8.9
3,B,bottom20,14,6,8.9


### S1 — merged bus stops

CTA lists most bus stops as a pair, one per side of the street; SPTrans less often. Merging stops with the same normalized name within 50 m tests whether that convention drives the result.

In [14]:
ab.test_s1(out, S1)

,method,spearman_with_primary,top10_overlap,bras_rank_primary,bras_rank_merged,max_rank_shift
0,A,0.9956,9,106,97,16
1,B,0.9943,9,23,24,18


## 5. Application: the Chicago area most similar to Brás

The project's goal is to use the model to choose a Chicago Community Area similar to Brás. With the agreed one-number index, **the most similar area is the one whose index value is closest to Brás's**: $\text{gap}(u) = |I(u) - I(\text{Brás})|$, smallest first. Each method chooses on its own index.

Diagnostic (not part of the agreed model): `profile_distance` is the Euclidean distance between the six scaled factors of the area and of Brás. A large profile distance means the index matched a total built from a different mix of factors.

In [15]:
match = ab.match_chicago(res)
for m in ('A', 'B'):
    print('Method', m); display(match[match.method == m].head(5))

Method A


,method,unit_id,name,index,reference,reference_index,index_gap,rank_by_index_gap,profile_distance,rank_by_profile_distance
70,A,CHI:71,AUBURN GRESHAM,0.8280,SP:10,0.8334,0.0054,1,0.3698,44
50,A,CHI:51,SOUTH DEERING,0.8229,SP:10,0.8334,0.0105,2,0.5255,74
30,A,CHI:31,LOWER WEST SIDE,0.8215,SP:10,0.8334,0.0119,3,0.3057,7
22,A,CHI:23,HUMBOLDT PARK,0.8548,SP:10,0.8334,0.0214,4,0.3570,31
60,A,CHI:61,NEW CITY,0.8644,SP:10,0.8334,0.0310,5,0.3905,55


Method B


,method,unit_id,name,index,reference,reference_index,index_gap,rank_by_index_gap,profile_distance,rank_by_profile_distance
5,B,CHI:06,LAKE VIEW,1.8522,SP:10,1.9975,0.1453,1,0.4265,7
76,B,CHI:77,EDGEWATER,1.7057,SP:10,1.9975,0.2918,2,0.4109,5
2,B,CHI:03,UPTOWN,1.6469,SP:10,1.9975,0.3506,3,0.5242,29
27,B,CHI:28,NEAR WEST SIDE,1.6152,SP:10,1.9975,0.3823,4,0.3937,2
33,B,CHI:34,ARMOUR SQUARE,1.5897,SP:10,1.9975,0.4078,5,0.5328,33


The areas whose six-factor profile is closest to Brás, with their position in the index ranking:

In [16]:
match.sort_values(['method', 'rank_by_profile_distance']).groupby('method').head(3)

,method,unit_id,name,index,reference,reference_index,index_gap,rank_by_index_gap,profile_distance,rank_by_profile_distance
4,A,CHI:05,NORTH CENTER,0.6891,SP:10,0.8334,0.1443,26,0.2638,1
6,A,CHI:07,LINCOLN PARK,1.1368,SP:10,0.8334,0.3035,54,0.2679,2
76,A,CHI:77,EDGEWATER,0.7531,SP:10,0.8334,0.0803,12,0.2901,3
23,B,CHI:24,WEST TOWN,1.5510,SP:10,1.9975,0.4465,7,0.3483,1
27,B,CHI:28,NEAR WEST SIDE,1.6152,SP:10,1.9975,0.3823,4,0.3937,2
21,B,CHI:22,LOGAN SQUARE,1.4976,SP:10,1.9975,0.4999,9,0.3958,3


In [17]:
F = ab.FACTORS
picks = {'A': match[(match.method == 'A') & (match.rank_by_index_gap == 1)].unit_id.iloc[0],
         'B': match[(match.method == 'B') & (match.rank_by_index_gap == 1)].unit_id.iloc[0]}
prof = pd.DataFrame({f'{m} scaled: {lab}': [res.at[u, f'{m}_scaled_{f}'] for f in F]
                     for m in ('A', 'B') for lab, u in (('Brás', 'SP:10'), (NAMES[picks[m]], picks[m]))}, index=F)
prof.round(3)

,A scaled: Brás,A scaled: AUBURN GRESHAM,B scaled: Brás,B scaled: LAKE VIEW
C,0.344,0.040,0.410,0.132
R,0.121,0.127,0.305,0.429
H,0.089,0.252,0.302,0.419
V,0.101,0.056,0.301,0.310
N,0.092,0.166,0.593,0.350
L,0.086,0.188,0.086,0.213


## 6. Reproducibility check against the published report

The tables below were written by `build_simple_index_ab.py` and summarized in the methodology report. This cell asserts that the notebook reproduces them exactly.

In [18]:
PUB = ROOT / 'analysis/results/SP_CHI/simple_index_ab_2026_10_01/tables'
pd.testing.assert_frame_equal(t1.reset_index(drop=True), pd.read_csv(PUB / 't1_boundary_invariance.csv'), check_dtype=False)
pd.testing.assert_frame_equal(t2.reset_index(drop=True), pd.read_csv(PUB / 't2_size_dependence.csv'), check_dtype=False)
pub_idx = pd.read_csv(PUB / 'index_A_B.csv').set_index('unit_id')
for c in ['A_index', 'B_index', 'A_rank', 'B_rank']:
    pd.testing.assert_series_equal(res[c], pub_idx[c], check_dtype=False, check_names=False)
pd.testing.assert_frame_equal(match.reset_index(drop=True), pd.read_csv(PUB / 'bras_chicago_match.csv'), check_dtype=False)
pub_t3 = pd.read_csv(PUB / 't3_agreement.csv').iloc[0]
assert all(str(pub_t3[k]) == str(v) for k, v in t3.items() if k in pub_t3), 'T3 differs'
print('Notebook reproduces the published tables exactly.')

Notebook reproduces the published tables exactly.


## 7. How to interpret the results

- **T1 is the decisive test.** A factor that fails it changes value when boundaries are redrawn, so it cannot be read as urban intensity. Under the pre-registered criterion, method A's count factors (C, R, H, N) and its index fail and method B passes.
- **T2 and T3 show the practical consequence:** whether A's index rises with unit size, and how differently A and B rank the same places, including Brás.
- **Application:** each method picks the Chicago area with the closest index value to Brás. The profile distance shows whether that area also resembles Brás factor by factor; a one-number index can match totals built from different mixes, which is the information loss accepted for this first sketch.
- **What B does not fix:** both methods use the same sources, so uneven Overture coverage across districts (3.2%–35.2% of CNEFE establishment addresses, after subtracting chance matches) affects both, and the cross-city comparison (T4) cannot be validated with any available reference.
- **Not changed in B, by agreement:** average street length enters positively although longer segments usually mean a coarser network; min–max scaling lets single extreme units set each scale.